In [12]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
    Trainer
    ,TrainingArguments
)
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTTrainer
import os 
from datasets import load_dataset
import pandas as pd


In [13]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/home/ltnga/NguyenTrinh/outputs_stage3/final_model",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2024.12.2: Fast Qwen2 patching. Transformers:4.46.3.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.138 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2024.12.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
        

In [15]:
import json
import time
ids = []
questions = []
choices_A = []
choices_B = []
choices_C = []
choices_D = []
choices_E = []
answers=[]
with open('/home/ltnga/ITDSIU21079/VMLU/vmlu_v1.5/valid.jsonl', "r", encoding="utf-8") as f:
            lines = f.readlines()
            for line in lines:
                data = json.loads(line)
                ids.append(data["id"])
                questions.append(data["question"])
                answers.append(data["answer"])
                choices = data["choices"]
                try:
                    choices_A.append(choices[0])
                except:
                    choices_A.append('')
                try:
                    choices_B.append(choices[1])
                except:
                    choices_B.append('')
                try:
                    choices_C.append(choices[2])
                except:
                    choices_C.append('')
                try:
                    choices_D.append(choices[3])
                except:
                    choices_D.append('')
                try:
                    choices_E.append(choices[4])
                except:
                    choices_E.append('')

In [16]:
questions[1]

'Những yếu tố nào sau đây có thể dẫn đến thâm hụt cán cân thương mại của một nước:'

In [17]:
df = pd.DataFrame({
        "id": ids,
        "prompt": questions,
        "A": choices_A,
        "B": choices_B,
        "C": choices_C,
        "D": choices_D,
        "anwser":answers
    })

In [18]:
df["anwser"].value_counts()

anwser
C    202
D    191
A    181
B    170
Name: count, dtype: int64

In [19]:
from string import Template
preamble = \
        'Chỉ đưa ra chữ cái đứng trước câu trả lời đúng (A, B, C, D) của câu hỏi trắc nghiệm sau: '

template = Template('$preamble\n\n$prompt\n\n $a\n $b\n $c\n $d \nĐáp án:')

def format_input(df, idx):
    prompt = df.loc[idx, 'prompt']
    a = df.loc[idx, 'A']
    b = df.loc[idx, 'B']
    c = df.loc[idx, 'C']
    d = df.loc[idx, 'D']
    # e = df.loc[idx, 'E']

    input_text = template.substitute(
        preamble=preamble, prompt=prompt, a=a, b=b, c=c, d=d)
        
    return input_text

In [ ]:
from tqdm import tqdm
FastLanguageModel.for_inference(model)
start = time.time()
for idx in df.index:
    
    inputs = tokenizer(format_input(df, idx), return_tensors="pt", return_token_type_ids=False).to(device)
    outputs = model.generate(**inputs, pad_token_id=tokenizer.eos_token_id, max_new_tokens=1)
    answer_decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    last_element = answer_decoded[-1]
    answer = last_element.split()[-1]
    answers.append(answer)

end = time.time()
duration = end - start
print('Time taken for running inference: ', duration)
df["answer_bot"]=answer


    

Time taken for running inference:  80.80754494667053


In [21]:
#Base model Phần trăm đúng: 27.15%
#Stage 1 Phần trăm đúng: 25.67%
#Stage 2 Phần trăm đúng: 25.67%
#Stage 3 Phần trăm đúng: 25.67%
#Merge Phần trăm đúng: 22.85%
df['is_correct'] = df['anwser'] == df['answer_bot']  # Tạo cột kiểm tra đúng/sai
correct_count = df['is_correct'].sum()               # Tổng số câu đúng
total_count = len(df)                                # Tổng số câu hỏi

# Tính % đúng
accuracy = (correct_count / total_count) * 100
print(f"Phần trăm đúng: {accuracy:.2f}%")

Phần trăm đúng: 25.67%
